# Pokémon Card Identifier — Training Notebook

**Run on Google Colab with a GPU runtime.**

Cells in order:
1. GPU check
2. Install dependencies
3. Mount Drive + clone repo
4. Get card data (Drive cache → CDN fallback)
5. Train embedding model
6. Build FAISS index
7. Inference validation

## Cell 1 — Runtime / GPU check

In [ ]:
import torch, platform, subprocess

print(f"Python:  {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:     {gpu} ({mem_gb:.1f} GB)")
else:
    print("WARNING: No GPU detected — training will be very slow.")
    print("Runtime → Change runtime type → T4 GPU")

## Cell 2 — Install missing dependencies

Colab pre-installs torch, torchvision, torchaudio, pandas, tqdm, and Pillow.
We only need to add `faiss-cpu` and `httpx`, which Colab doesn't ship with.

In [ ]:
# Colab pre-installs torch/torchvision/torchaudio — don't reinstall them.
# pandas, tqdm, and Pillow are also part of Colab's base image.
# Only install packages Colab doesn't ship with.
!pip install -q \
    'faiss-cpu>=1.10.0' \
    'httpx>=0.28.1'

import torch, torchvision
print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print("Dependencies installed.")

## Cell 3 — Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
from pathlib import Path

# CONFIG
GITHUB_REPO   = "https://github.com/jbailey7/pokemon-card-pricing.git"
GITHUB_BRANCH = "model-training"
REPO_DIR      = Path("/content/pokemon-card-pricing")
DRIVE_DIR     = Path("/content/drive/MyDrive/pokemon")  # persistent storage
CHECKPOINT_DIR = DRIVE_DIR / "checkpoints"
INDEX_DIR      = DRIVE_DIR / "index"
DATA_DIR       = REPO_DIR / "data"
IMAGES_DIR     = DATA_DIR / "images"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Clone or update repo
if REPO_DIR.exists():
    print("Repo already cloned — pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    print(f"Cloning {GITHUB_REPO} (branch: {GITHUB_BRANCH})...")
    subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(REPO_DIR)],
        check=True,
    )

# Add repo to Python path so `model` package is importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Branch:      {subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current']).decode().strip()}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Index:       {INDEX_DIR}")

## Cell 4 — Get card data

Strategy (in order):
1. If `cards.csv` already exists in Drive → symlink it (fastest)
2. Else run `data/download_cards.py` (clones metadata JSON + downloads images from CDN)

For a **test run**, set `LIMIT = 500`.  
For **full training**, set `LIMIT = None`.

In [ ]:
import subprocess
from pathlib import Path

#  CONFIG 
LIMIT = None   # set to e.g. 500 for a quick test, None for full download

DRIVE_CSV    = DRIVE_DIR / "cards.csv"
DRIVE_IMAGES = DRIVE_DIR / "images"
LOCAL_CSV    = DATA_DIR / "cards.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_CSV.exists() and DRIVE_IMAGES.exists():
    # Fast path: link Drive cache into repo data/ directory
    print("Using cached data from Drive...")
    if not LOCAL_CSV.exists():
        LOCAL_CSV.symlink_to(DRIVE_CSV)
    drive_images_link = DATA_DIR / "images"
    if not drive_images_link.exists():
        drive_images_link.symlink_to(DRIVE_IMAGES)
    print(f"  cards.csv: {LOCAL_CSV}")
    print(f"  images/:   {drive_images_link}")
else:
    # Download path: clone JSON repo + download images from CDN
    print("Downloading card data from CDN...")
    cmd = ["python", "data/download_cards.py"]
    if LIMIT is not None:
        cmd += ["--limit", str(LIMIT)]
    subprocess.run(cmd, check=True)

    # Copy to Drive for future sessions
    import shutil
    if LOCAL_CSV.exists() and not DRIVE_CSV.exists():
        print("Copying cards.csv to Drive...")
        shutil.copy(LOCAL_CSV, DRIVE_CSV)
    if IMAGES_DIR.exists() and not DRIVE_IMAGES.exists():
        print("Copying images to Drive (this takes a few minutes)...")
        shutil.copytree(IMAGES_DIR, DRIVE_IMAGES)

# Sanity check
import pandas as pd
df = pd.read_csv(LOCAL_CSV)
n_images = sum(1 for p in IMAGES_DIR.rglob("*_hires.png")) if IMAGES_DIR.exists() else 0
print(f"\nCards in CSV:    {len(df):,}")
print(f"Images on disk:  {n_images:,}")

## Cell 5 — Train embedding model

Saves `best_model.pt` and `last_model.pt` to Drive after each epoch.

In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so `model` is importable
REPO_DIR = Path("/content/pokemon-card-pricing")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from model.train import train, parse_args

# CONFIG
EPOCHS_TOTAL  = 30      # full training; set to 2 for smoke test
FROZEN_EPOCHS = 5       # epochs with backbone frozen
TRAIN_LIMIT   = None    # None = all cards; set to 500 for smoke test

DATA_DIR       = REPO_DIR / "data"
CHECKPOINT_DIR = Path("/content/drive/MyDrive/pokemon/checkpoints")

args_list = [
    "--data-dir",        str(DATA_DIR),
    "--checkpoint-dir",  str(CHECKPOINT_DIR),
    "--epochs-total",    str(EPOCHS_TOTAL),
    "--frozen-epochs",   str(FROZEN_EPOCHS),
]
if TRAIN_LIMIT is not None:
    args_list += ["--limit", str(TRAIN_LIMIT)]

print("Starting training...")
print("model.train", " ".join(args_list))

_orig_argv = sys.argv
sys.argv = ["model.train"] + args_list
try:
    args = parse_args()
finally:
    sys.argv = _orig_argv

train(args)

## Cell 6 — Build FAISS index

In [ ]:
import sys
from pathlib import Path

REPO_DIR       = Path("/content/pokemon-card-pricing")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/pokemon/checkpoints")
INDEX_DIR      = Path("/content/drive/MyDrive/pokemon/index")
DATA_DIR       = REPO_DIR / "data"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from model.build_index import build_index, parse_args as parse_index_args

best_ckpt = CHECKPOINT_DIR / "best_model.pt"
assert best_ckpt.exists(), f"Checkpoint not found: {best_ckpt}"

args_list = [
    "--checkpoint",  str(best_ckpt),
    "--data-dir",    str(DATA_DIR),
    "--output-dir",  str(INDEX_DIR),
]

print("Building FAISS index...")

_orig_argv = sys.argv
sys.argv = ["model.build_index"] + args_list
try:
    args = parse_index_args()
finally:
    sys.argv = _orig_argv

build_index(args)

print(f"\nIndex files:")
for f in INDEX_DIR.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

## Cell 7 — Inference validation

Query `data/images/base1/4_hires.png` (Charizard, Base Set #4).  
Top-1 result should be `base1-4` — Charizard.

In [ ]:
from PIL import Image
from model.inference import CardIdentifier

# Resolve paths (handle symlinks)
identifier = CardIdentifier(
    checkpoint=str(CHECKPOINT_DIR / "best_model.pt"),
    index_dir=str(INDEX_DIR),
)

# Test with Charizard
test_img_path = DATA_DIR / "images" / "base1" / "4_hires.png"
assert test_img_path.exists(), f"Test image not found: {test_img_path}"

img = Image.open(test_img_path)
results = identifier.predict(img, k=3)

print("Query: Charizard (base1-4)\n")
print(f"{'Rank':<6} {'Score':<8} {'ID':<20} {'Name':<25} {'Set'}")
print("-" * 80)
for r in results:
    print(f"{r['rank']:<6} {r['score']:<8.4f} {r['id']:<20} {r['name']:<25} {r['set_name']}")

top1 = results[0]
if top1["id"] == "base1-4":
    print("\n✓ Top-1 correct: Charizard base1-4")
else:
    print(f"\n✗ Top-1 WRONG: got {top1['id']} ({top1['name']})")